Este notebook irá montar um tabela com dados dos Municipios Brasileiros e a quantidade da poplulação por grupos de idade

- 2 a 14 anos
- 15 a 59 anos
- 60 anos ou mais 




In [1]:
import os, sys, requests
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Recupera as informações da idade da população - IBGE via API SIDRA. 

Tabela 10131 - Pessoas residentes de 2 anos ou mais de idade por grupo de idade e existência de deficiência

A URL é definida para buscar os dados de população urbana e rural por município.

In [9]:
url_populacao_urbana_rural = "https://apisidra.ibge.gov.br/values/t/10131/n6/all/v/11852/p/all/c58/3302,73770,104578/c839/46583"
try:
    # 1. Faz a requisição HTTP para a API do IBGE
    response = requests.get(url_populacao_urbana_rural)
    response.raise_for_status()  # Garante que a requisição funcionou (Status 200)

    # 2. Converte a resposta bruta para o formato JSON
    populacao_idade_json = response.json()

    print(len(populacao_idade_json))
except Exception as e:
    print(e)    

16711


In [10]:


cabecalho    = populacao_idade_json[0]
linhas_dados = populacao_idade_json[1:]

mapeamento_colunas = {k: v for k, v in cabecalho.items()}
dados_processados = []
for linha in linhas_dados:
    nova_linha = {mapeamento_colunas[chave]: valor for chave, valor in linha.items() if chave in mapeamento_colunas}
    dados_processados.append(nova_linha)

df_populacao_idade = spark.createDataFrame(dados_processados)



In [11]:
df_populacao_idade.printSchema()
df_populacao_idade.show(10, False)

root
 |-- Ano: string (nullable = true)
 |-- Ano (Código): string (nullable = true)
 |-- Existência de deficiência: string (nullable = true)
 |-- Existência de deficiência (Código): string (nullable = true)
 |-- Grupo de idade: string (nullable = true)
 |-- Grupo de idade (Código): string (nullable = true)
 |-- Município: string (nullable = true)
 |-- Município (Código): string (nullable = true)
 |-- Nível Territorial: string (nullable = true)
 |-- Nível Territorial (Código): string (nullable = true)
 |-- Unidade de Medida: string (nullable = true)
 |-- Unidade de Medida (Código): string (nullable = true)
 |-- Valor: string (nullable = true)
 |-- Variável: string (nullable = true)
 |-- Variável (Código): string (nullable = true)

+----+------------+-------------------------+----------------------------------+---------------+-----------------------+--------------------------+------------------+-----------------+--------------------------+-----------------+--------------------------+----

In [15]:
# Separa somente as colunas desejadas.
df_populacao_idade_ = \
    df_populacao_idade.select(
         F.col("Ano").alias("ano")
        ,F.col("Município (Código)").alias("codigo_municipio")
        ,F.col("Município").alias("nome_municipio")
        ,F.col("Grupo de idade (Código)").alias("codigo_grupo_idade")
        ,F.col("Grupo de idade").alias("descricao_grupo_idade")
        ,F.col("Valor").alias("quantidade_populacao_idade"))


In [ ]:
df_populacao_idade_.show(10, False)

+----+----------------+--------------------------+------------------+---------------------+--------------------------+
|ano |codigo_municipio|nome_municipio            |codigo_grupo_idade|descricao_grupo_idade|quantidade_populacao_idade|
+----+----------------+--------------------------+------------------+---------------------+--------------------------+
|2022|1100015         |Alta Floresta D'Oeste - RO|3302              |60 anos ou mais      |3125                      |
|2022|1100015         |Alta Floresta D'Oeste - RO|73770             |2 a 14 anos          |4022                      |
|2022|1100015         |Alta Floresta D'Oeste - RO|104578            |15 a 59 anos         |13742                     |
|2022|1100023         |Ariquemes - RO            |3302              |60 anos ou mais      |11067                     |
|2022|1100023         |Ariquemes - RO            |73770             |2 a 14 anos          |18474                     |
|2022|1100023         |Ariquemes - RO           

### Faz a transposição das informações de linhas para colunas 

In [16]:
df_populacao_idade_.createOrReplaceTempView("temp_municipios_idade")
query = \
    """Select ano, codigo_municipio, descricao_grupo_idade, quantidade_populacao_idade
         from temp_municipios_idade
    """
df_populacao_grupo_idade = spark.sql(query)

# df_municipio_populacao_urbana_rural.printSchema()

# df_municipio_populacao_urbana_rural.show(10,False)

# Executando o Pivot para criar as colunas "Urbana" e "Rural" com os valores correspondentes
df_populacao_idade_pivot = \
    df_populacao_grupo_idade \
        .groupBy("ano", "codigo_municipio" ) \
        .pivot("descricao_grupo_idade", ["2 a 14 anos", "15 a 59 anos", "60 anos ou mais"]) \
        .agg(F.first("quantidade_populacao_idade"))

# Exibindo o resultado
df_populacao_idade_pivot.show(truncate=False)


+----+----------------+-----------+------------+---------------+
|ano |codigo_municipio|2 a 14 anos|15 a 59 anos|60 anos ou mais|
+----+----------------+-----------+------------+---------------+
|2022|1100015         |4022       |13742       |3125           |
|2022|1100023         |18474      |64751       |11067          |
|2022|1100031         |990        |3226        |1029           |
|2022|1100049         |15323      |57836       |11583          |
|2022|1100056         |2876       |10178       |2334           |
|2022|1100064         |2675       |9889        |2689           |
|2022|1100072         |1353       |4899        |1089           |
|2022|1100080         |2797       |8110        |1330           |
|2022|1100098         |5406       |19181       |3896           |
|2022|1100106         |9266       |24564       |4246           |
|2022|1100114         |9343       |32761       |7031           |
|2022|1100122         |23431      |81877       |15716          |
|2022|1100130         |64

In [16]:
df_munic_pivot_situacao.printSchema()

root
 |-- ano: string (nullable = true)
 |-- codigo_municipio: string (nullable = true)
 |-- Urbana: string (nullable = true)
 |-- Rural: string (nullable = true)



In [33]:
# Este dataframe contém informações sobre o bioma predominante, a grande região, e a população urbana e rural de cada município. 
# A junção é feita com base no código do município, utilizando um join à esquerda para garantir que todos os municípios do DataFrame de biomas sejam mantidos, 
# mesmo que não existam dados de população urbana/rural correspondentes.

df_munic_sit_bioma = \
    (df_bioma_grande_regiao.alias('b')
        .join(df_munic_pivot_situacao.alias('s')
             ,F.col('b.codigo_municipio') == F.col('s.codigo_municipio')
             ,"left")
        .select('b.codigo_municipio'
               ,'b.municipio'
               ,'b.UF_municipio'
               ,'b.bioma'
               ,'b.grande_regiao'
               ,'s.Urbana'
               ,'s.Rural') )
        
df_munic_sit_bioma.show(truncate=False)

#   .where("s.codigo_municipio is null or (s.Urbana = '0' or s.Rural = '0')") \



+----------------+------------------------+------------+--------+-------------+------+-----+
|codigo_municipio|municipio               |UF_municipio|bioma   |grande_regiao|Urbana|Rural|
+----------------+------------------------+------------+--------+-------------+------+-----+
|1100114         |Jaru                    |RO          |Amazônia|Norte        |39807 |10784|
|1100288         |Rolim de Moura          |RO          |Amazônia|Norte        |47264 |9142 |
|1100189         |Pimenta Bueno           |RO          |Amazônia|Norte        |28710 |6369 |
|1100049         |Cacoal                  |RO          |Amazônia|Norte        |71907 |14980|
|1100262         |Rio Crespo              |RO          |Amazônia|Norte        |1438  |2033 |
|1100155         |Ouro Preto do Oeste     |RO          |Amazônia|Norte        |27701 |7343 |
|1100205         |Porto Velho             |RO          |Amazônia|Norte        |426299|34135|
|1100064         |Colorado do Oeste       |RO          |Amazônia|Norte

In [34]:
# Salva o Dataframe como csv
(df_munic_sit_bioma
    .toPandas()
    .to_csv(r"C:\Marco Conti\Projetos\MAIS-v2\dados\tb_munic_sit_bioma.csv"
           ,index=False
           ,sep=";"))



c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [ ]:
# import subprocess

# resultado = subprocess.run(
#     r'"c:\Marco Conti\Projetos\MAIS-v2\.venv\Scripts\python.exe" -m pip install PyArrow', # trocar show por install 
#     shell=True,
#     capture_output=True,
#     text=True
# )

# print(resultado.stdout)
# print(resultado.stderr)
# print(resultado.returncode)

  Using cached pyarrow-25.0.0-cp311-cp311-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-25.0.0-cp311-cp311-win_amd64.whl (27.8 MB)


0
